<a href="https://colab.research.google.com/github/data4class/Teaching/blob/main/Deep_Demo_3_2025_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!wget -O jindal.trade https://drive.google.com/uc?id=1QcKzCrzGr3kqIYZZ_QTKaZeZVZWwHRaw&export=download
!wget -O jindal.order https://drive.google.com/uc?id=10D7Og3wYfFmU8UfnXEPHPlVShXCS2rma&export=download
## Make adjustment in read based on new file

--2025-10-12 03:18:38--  https://drive.google.com/uc?id=1QcKzCrzGr3kqIYZZ_QTKaZeZVZWwHRaw
Resolving drive.google.com (drive.google.com)... 74.125.137.100, 74.125.137.139, 74.125.137.138, ...
Connecting to drive.google.com (drive.google.com)|74.125.137.100|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QcKzCrzGr3kqIYZZ_QTKaZeZVZWwHRaw [following]
--2025-10-12 03:18:38--  https://drive.usercontent.google.com/download?id=1QcKzCrzGr3kqIYZZ_QTKaZeZVZWwHRaw
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.137.132, 2607:f8b0:4023:c03::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.137.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 174977 (171K) [application/octet-stream]
Saving to: ‘jindal.trade’

jindal.trade        100%[===================>] 170.88K  --.-KB/s    in 0.05s   

2025-10-12 03:18:40 (3.44 MB/s) - 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# TODO: Change these parse programs. Change '|' to ',' and skip header,
# TODO: and use correct index for columns. Same for trade.
def parse_order_line(line):
    """Parse a single order line"""
    parts = line.strip().split('|')
    if len(parts) < 12:
        return None

    try:
        return {
            'order_id': parts[1],
            'side': parts[2],
            'action': int(parts[3]),
            'time': int(parts[4]),
            'price': int(parts[5]),
            'quantity': int(parts[6]),
            'is_market': parts[8] == 'Y',  # Market order flag at index 8
            'is_ioc': parts[9] == 'Y'  # IOC flag at index 9
        }
    except (ValueError, IndexError) as e:
        return None

def parse_trade_line(line):
    """Parse a single trade line
    Format: RM|CASH|2021012050180245|84908554574642|N5|00106650|00000003|1200000000893059|1|3|1200000000212373|1
    Fields: 0=RM, 1=CASH, 2=?, 3=timestamp, 5=price, 6=quantity, rest=order IDs
    """
    parts = line.strip().split('|')
    if len(parts) < 7:
        return None

    try:
        return {
            'time': int(parts[3]),      # Timestamp at index 3
            'price': int(parts[5]),     # Price at index 5
            'quantity': int(parts[6])   # Quantity at index 6
        }
    except (ValueError, IndexError):
        return None

def add_to_order_book(order_book, side, price, quantity):
    """Add order to order book"""
    if price in order_book[side]:
        order_book[side][price] += quantity
    else:
        order_book[side][price] = quantity

def remove_from_order_book(order_book, side, price, quantity):
    """Remove order from order book"""
    if price in order_book[side]:
        order_book[side][price] -= quantity
        if order_book[side][price] <= 0:
            del order_book[side][price]

def update_order_book(order_book, side, old_price, new_price, new_quantity):
    """Update order in order book (action 4)"""
    # Remove old order
    if old_price in order_book[side]:
        del order_book[side][old_price]
    # Add new order
    if new_quantity > 0:
        order_book[side][new_price] = new_quantity

def generate_snapshots(order_file_path, trade_file_path, interval_seconds=60):
    """Generate order book snapshots at regular intervals with next trade price as target"""

    interval_ziffies = interval_seconds * (2 ** 16)  # Convert to ziffies

    order_book = {"B": {}, "S": {}}
    snapshots = []
    targets = []

    # Track orders by ID for updates
    order_tracker = {}  # order_id -> (side, price, quantity)

    # Read all orders and trades into memory
    print("Reading order file...")
    with open(order_file_path, 'r') as f:
        order_lines = [line for line in f if line.strip()]

    print("Reading trade file...")
    with open(trade_file_path, 'r') as f:
        trade_lines = [line for line in f if line.strip()]

    print(f"Read {len(order_lines)} order lines and {len(trade_lines)} trade lines")

    # Debug: Show first few lines
    if order_lines:
        print(f"\nFirst order line sample: {order_lines[0][:100]}")
    if trade_lines:
        print(f"First trade line sample: {trade_lines[0][:100]}")

    # Parse orders and trades
    print("\nParsing orders...")
    orders = []
    parse_errors = 0
    for i, line in enumerate(order_lines):
        order = parse_order_line(line)
        if order:
            orders.append(order)
        else:
            parse_errors += 1
            if parse_errors <= 3:  # Show first 3 errors
                print(f"  Failed to parse order line {i+1}: {line[:80]}")

    if parse_errors > 3:
        print(f"  ... and {parse_errors - 3} more parsing errors")

    print("\nParsing trades...")
    trades = []
    parse_errors = 0
    for i, line in enumerate(trade_lines):
        trade = parse_trade_line(line)
        if trade:
            trades.append(trade)
        else:
            parse_errors += 1
            if parse_errors <= 3:  # Show first 3 errors
                print(f"  Failed to parse trade line {i+1}: {line[:80]}")

    if parse_errors > 3:
        print(f"  ... and {parse_errors - 3} more parsing errors")

    if not orders or not trades:
        print("ERROR: No valid orders or trades found")
        return [], []

    print(f"Parsed {len(orders)} orders and {len(trades)} trades")

    # Sort by time
    orders.sort(key=lambda x: x['time'])
    trades.sort(key=lambda x: x['time'])

    # Initialize
    order_idx = 0
    trade_idx = 0

    if orders:
        current_time = orders[0]['time']
        interval_end_time = current_time + interval_ziffies
    else:
        return [], []

    snapshot_count = 0

    while order_idx < len(orders) or trade_idx < len(trades):

        # Process all orders in current interval
        while order_idx < len(orders) and orders[order_idx]['time'] < interval_end_time:
            order = orders[order_idx]

            # Skip market orders and IOC orders (they don't stay in order book)
            if order['is_market'] or order['is_ioc']:
                order_idx += 1
                continue

            if order['action'] == 1:  # Add order
                add_to_order_book(order_book, order['side'], order['price'], order['quantity'])
                order_tracker[order['order_id']] = (order['side'], order['price'], order['quantity'])

            elif order['action'] == 3:  # Remove order
                remove_from_order_book(order_book, order['side'], order['price'], order['quantity'])
                if order['order_id'] in order_tracker:
                    del order_tracker[order['order_id']]

            elif order['action'] == 4:  # Update order
                # Remove old order if it exists
                if order['order_id'] in order_tracker:
                    old_side, old_price, old_quantity = order_tracker[order['order_id']]
                    remove_from_order_book(order_book, old_side, old_price, old_quantity)

                # Add new order
                add_to_order_book(order_book, order['side'], order['price'], order['quantity'])
                order_tracker[order['order_id']] = (order['side'], order['price'], order['quantity'])

            order_idx += 1

        # Advance trades to interval end
        while trade_idx < len(trades) and trades[trade_idx]['time'] < interval_end_time:
            trade_idx += 1

        # Find next trade after interval (this is our target)
        next_trade_price = None
        if trade_idx < len(trades):
            next_trade_price = trades[trade_idx]['price']

        # Save snapshot if we have valid data
        if next_trade_price is not None and (order_book["B"] or order_book["S"]):
            snapshot_copy = {
                "B": dict(order_book["B"]),
                "S": dict(order_book["S"])
            }
            snapshots.append(snapshot_copy)
            targets.append(next_trade_price)
            snapshot_count += 1

            if snapshot_count % 100 == 0:
                print(f"Generated {snapshot_count} snapshots...")

        # Move to next interval
        interval_end_time += interval_ziffies

        # Stop if no more data
        if order_idx >= len(orders) and trade_idx >= len(trades):
            break

    print(f"Total snapshots generated: {len(snapshots)}")
    return snapshots, targets
# EX: Rewrite above program to handle stoploss, iceberg orders.



def flatten_snapshot(snapshot):
    """Convert order book snapshot to fixed-length feature vector

    Critical: Always returns fixed-size vector with meaningful values.
    Uses 0 for both price AND quantity when no orders exist at that level.
    The NN learns that (price=0, qty=0) means "no order at this level".
    """
    max_depth = 10

    # Get sorted bid and ask prices
    bid_prices = sorted(snapshot['B'].keys(), reverse=True)[:max_depth]
    bid_quantities = [snapshot['B'][price] for price in bid_prices]

    ask_prices = sorted(snapshot['S'].keys())[:max_depth]
    ask_quantities = [snapshot['S'][price] for price in ask_prices]

    # Pad with ZEROS for both price and quantity when no orders exist
    # This is consistent: (price=0, qty=0) = no order at this level
    bid_prices = bid_prices + [0] * (max_depth - len(bid_prices))
    bid_quantities = bid_quantities + [0] * (max_depth - len(bid_quantities))
    ask_prices = ask_prices + [0] * (max_depth - len(ask_prices))
    ask_quantities = ask_quantities + [0] * (max_depth - len(ask_quantities))

    # Calculate additional features
    mid_price = 0
    spread = 0
    # Ex: Use weighted bid-ask spread
    if len(snapshot['B']) > 0 and len(snapshot['S']) > 0:
        best_bid = max(snapshot['B'].keys())
        best_ask = min(snapshot['S'].keys())
        mid_price = (best_bid + best_ask) / 2
        spread = best_ask - best_bid

    bid_volume = sum(snapshot['B'].values())
    ask_volume = sum(snapshot['S'].values())
    volume_imbalance = (bid_volume - ask_volume) / (bid_volume + ask_volume + 1e-8)

    # Weighted mid price (volume-weighted)
    weighted_bid_price = 0
    weighted_ask_price = 0
    if bid_volume > 0:
        weighted_bid_price = sum(p * q for p, q in snapshot['B'].items()) / bid_volume
    if ask_volume > 0:
        weighted_ask_price = sum(p * q for p, q in snapshot['S'].items()) / ask_volume

    # Combine all features
    features = (bid_prices + bid_quantities + ask_prices + ask_quantities +
                [mid_price, spread, bid_volume, ask_volume, volume_imbalance,
                 weighted_bid_price, weighted_ask_price])

    # Ex: Add more features.

    return features

# ======================== MAIN EXECUTION ========================

# File paths
# TODO: Here, change file names
order_file = "ltinfra.order"
trade_file = "LTINFRA.trade"

# Quick file check
print("\n=== Checking Files ===")
import os
if os.path.exists(order_file):
    print(f"✓ Order file found: {order_file}")
    with open(order_file, 'r') as f:
        first_line = f.readline()
        print(f"  First line: {first_line[:150]}")
        print(f"  Number of fields: {len(first_line.split('|'))}")
else:
    print(f"✗ Order file NOT found: {order_file}")

if os.path.exists(trade_file):
    print(f"✓ Trade file found: {trade_file}")
    with open(trade_file, 'r') as f:
        first_line = f.readline()
        print(f"  First line: {first_line[:150]}")
        print(f"  Number of fields: {len(first_line.split(','))}")
else:
    print(f"✗ Trade file NOT found: {trade_file}")

# Generate snapshots with 1-minute intervals
print("\n=== Generating Snapshots ===")
snapshots, prices = generate_snapshots(order_file, trade_file, interval_seconds=60)

if len(snapshots) < 50:
    print("ERROR: Not enough snapshots generated. Check your data files.")
    exit(1)

# Prepare data for neural network
print("\n=== Preparing Data ===")
X = np.array([flatten_snapshot(snapshot) for snapshot in snapshots])
y = np.array(prices)

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")

if len(y) == 0:
    print("\nERROR: No data to train on!")
    print("Possible issues:")
    print("1. Check if file paths are correct")
    print("2. Check if files have the expected format")
    print("3. Check if there are enough valid orders and trades")
    exit(1)

print(f"Price range: {y.min()} to {y.max()}")
print(f"Price mean: {y.mean():.2f}, std: {y.std():.2f}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

# Normalize the data
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

# Build neural network model
print("\n=== Building Model ===")
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

# Train the model
print("\n=== Training Model ===")
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train_scaled,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate the model
print("\n=== Evaluating Model ===")
test_loss, test_mae = model.evaluate(X_test_scaled, y_test_scaled, verbose=0)
print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

# Make predictions
y_pred_scaled = model.predict(X_test_scaled, verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_test_actual = scaler_y.inverse_transform(y_test_scaled)

# Calculate error statistics
errors = y_test_actual - y_pred
mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors**2))
mape = np.mean(np.abs(errors / (y_test_actual + 1e-8))) * 100

print(f"\n=== Prediction Statistics ===")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

# Create visualizations
print("\n=== Creating Visualizations ===")
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Training history
axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Predicted vs Actual scatter plot
axes[0, 1].scatter(y_test_actual, y_pred, alpha=0.5, s=20)
axes[0, 1].plot([y_test_actual.min(), y_test_actual.max()],
                [y_test_actual.min(), y_test_actual.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0, 1].set_xlabel('Actual Price')
axes[0, 1].set_ylabel('Predicted Price')
axes[0, 1].set_title('Predicted vs Actual Prices', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Prediction errors distribution
axes[1, 0].hist(errors.flatten(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1, 0].set_xlabel('Prediction Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
axes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Time series of predictions (sample)
sample_size = min(100, len(y_test_actual))
indices = np.arange(sample_size)
axes[1, 1].plot(indices, y_test_actual[:sample_size], 'o-', label='Actual',
                markersize=4, linewidth=1.5, alpha=0.7)
axes[1, 1].plot(indices, y_pred[:sample_size], 's-', label='Predicted',
                markersize=4, linewidth=1.5, alpha=0.7)
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Price')
axes[1, 1].set_title(f'Actual vs Predicted (First {sample_size} Test Samples)',
                     fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Training Complete! ===")


=== Checking Files ===
✗ Order file NOT found: ltinfra.order
✗ Trade file NOT found: LTINFRA.trade

=== Generating Snapshots ===
Reading order file...


FileNotFoundError: [Errno 2] No such file or directory: 'ltinfra.order'

#OOP Code to compute snapshots

In [ ]:
class Trade:
  def __init__(self, trade_file_path):
    self.tf = open(trade_file_path, 'r')

  def next_trade(self):
    try:
      self.transaction = self.tf.readline()
      self.tdata = self.transaction.split('|')
      self.time = int(self.tdata[3])
      self.price = int(self.tdata[5])
      self.qty = int(self.tdata[6])
    except IndexError:
      return False


class Order:
  def __init__(self, order_file_path):
    self.of = open(order_file_path, 'r')

  def next_order(self):
    try:
      self.orderline = self.of.readline()
      self.odata = self.orderline.split('|')
      self.time = int(self.odata[4])
      self.price = int(self.odata[5])
      self.side = self.odata[2]
      self.action = int(self.odata[3])
      self.quantity = int(self.odata[6])
    except IndexError:
      return False


class Snapshot:
  def __init__(self, order_file_path, trade_file_path):
    self.order = Order(order_file_path)
    self.trade = Trade(trade_file_path)
    self.DS = {"B": dict(), "S": dict()}

  def add_(self):
    if(self.DS[self.order.side].__contains__(self.order.price)):
      self.DS[self.order.side][self.order.price] += self.order.quantity
    else:
      self.DS[self.order.side][self.order.price] = self.order.quantity

  def remove_(self):
    self.DS[self.order.side][self.order.price] -= self.order.quantity

  def build(self):
    self.trade.next_trade()
    self.order.next_order()
    while (self.order.time < self.trade.time):
      if(self.order.action == 1):
        self.add_()
      elif(self.order.action == 3):
        self.remove_()
      else:
        pass
      self.order.next_order()
    self.order.of.close()
    self.trade.tf.close()
    return self.DS

lt_order_file = "ltinfra.order"
lt_trade_file = "LTINFRA.trade"
lt_snapshot = Snapshot(lt_order_file, lt_trade_file)
lt_snapshot.build()